# MindEye2 + LoRA — Colab driver

Adapts the pretrained shared-subject MindEye2 model to a held-out subject with ~1 hour
of fMRI, six different ways (frozen / BitFit / LoRA r=4,16,64 / full fine-tune), then
compares them statistically and shows images.

**Every cell is safe to re-run.** If the runtime disconnects, reconnect, run cells 1-2
again, and continue — each stage checks the manifest on Drive and skips finished work.

Runtime → Change runtime type → **GPU** (T4 is fine to start).

## 1. Clone / update the repo

Mounts Drive and clones the repo, or pulls if it is already there.

In [ ]:
import os, sys

GITHUB_USER = "<your-username>"      # <-- change this
REPO_NAME   = "mindeye2-lora"
REPO        = f"/content/{REPO_NAME}"

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO}

os.chdir(REPO)

# Fail here with a clear message rather than three cells later on a cryptic ImportError
assert os.path.isdir(f"{REPO}/src/mindeye_lora"), (
    f"{REPO}/src/mindeye_lora not found. If GitHub shows a nested "
    f"{REPO_NAME}/{REPO_NAME}/ folder, the contents were uploaded one level too deep."
)
print("repo ready:", os.getcwd())

In [ ]:
!bash setup/colab_setup.sh

Installs come from the Drive-backed pip cache: 2-4 minutes the first time, under a
minute afterwards. The script resolves `dalle2-pytorch`'s transitive imports
automatically and ends with `dalle2_pytorch OK` / `environment ready`.

Colab sometimes needs a kernel restart after installs. If cell 2 fails on imports, run
this, then continue from cell 2 — nothing downloaded is lost.

In [ ]:
# import IPython; IPython.Application.instance().kernel.do_shutdown(True)

## 2. Attach to the workspace

**Run this first after any kernel restart or reconnect.** `main`, the Drive root and the
working directory are all lost on restart; the usual symptom is
`NameError: name 'main' is not defined`.

Setting `MINDEYE_LORA_ROOT` matters: without it the workspace silently falls back to
local disk, and every cached download disappears when the session ends.

In [ ]:
import os, sys

REPO = "/content/mindeye2-lora"
os.chdir(REPO)
os.environ["MINDEYE_LORA_ROOT"] = "/content/drive/MyDrive/mindeye2_lora"
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")

from mindeye_lora.cli import main

CONFIG = "configs/smoke.yaml"     # switch to configs/colab_t4.yaml for the real run

main(["setup", "--config", CONFIG])

## 3. Which config

| config | runtime | time |
|---|---|---|
| `configs/smoke.yaml` | any GPU | ~30 min — **start here** |
| `configs/colab_t4.yaml` | T4 (free) | several hours across sessions |
| `configs/a100_paper_scale.yaml` | A100 (Pro) | also `!pip install bitsandbytes` |

The smoke config runs 3 arms for 10 epochs and exercises every stage, so problems
surface in minutes rather than hours. Nothing it downloads is wasted — assets and CLIP
embeddings are cached on Drive and reused by the full run.

To switch, edit `CONFIG` in the cell above and re-run it.

Global flags work on either side of the subcommand, so `main(["train", "--config",
CONFIG])` and `main(["--config", CONFIG, "train"])` are equivalent.

## 4. Everything, in one resumable command

Runs assets → precompute → train → predict → recon → evaluate → compare → report,
skipping anything already finished. Re-run it verbatim after any disconnect.

Prefer to watch stage by stage? Skip this cell and use sections 5-13 instead.

In [ ]:
main(["run-all", "--config", CONFIG])

## 5. Assets

Downloads only what is needed. The 22 GB COCO image file is **sliced remotely over
HTTPS** — just the ~1,750 rows this experiment touches. The 2.86 GB pretrained
checkpoint is slimmed to weights-only and the original deleted.

First run: 10-20 minutes. Afterwards: instant. Watch for repeated `fetched N/M rows`.

In [ ]:
main(["assets", "--config", CONFIG])

## 6. Verify the pretrained weights

**The stage most likely to fail, and the cheapest place to find out.** It builds the
model, loads the shared-subject checkpoint, and refuses to continue if any non-ridge
parameter is missing — which would quietly turn "fine-tuning" into "training from
scratch" and invalidate the whole comparison.

Look for `skipped 7 pretrained ridge tensors`: those are subjects 2-8's subject-specific
layers, correctly left behind so subject 1 gets a fresh one.

In [ ]:
main(["verify", "--config", CONFIG])

## 7. Precompute CLIP embeddings

Embeds each stimulus once with OpenCLIP ViT-bigG/14 and caches the 256x1664 token
embeddings in fp16. This keeps the 2.5 GB vision tower out of memory during training,
which is what makes the T4 config fit.

In [ ]:
main(["precompute", "--config", CONFIG])

## 8. Train

One run per (arm, seed). All arms share data order, schedule and starting weights; only
the trainable parameter set differs. Watch the `trainable` counts differ by orders of
magnitude — that line is the experiment in miniature.

State is saved to Drive every 10 minutes and at each epoch boundary, and
`time_budget_min` stops cleanly before a session is likely to be reclaimed. Re-run this
cell next session to continue.

In [ ]:
main(["train", "--config", CONFIG])

In [ ]:
# where things stand — worth running after any disconnect
main(["status", "--config", CONFIG])

## 9. Predict

Runs each trained model over the test set and caches its predicted CLIP embeddings,
sampled through the diffusion prior.

In [ ]:
main(["predict", "--config", CONFIG])

## 10. Images

Two paths, and this cell always produces something.

**Retrieval fallback** (default): the nearest test-set images to each predicted
embedding, top-3, with correct hits outlined and the true rank annotated. Seconds, no
downloads. These are **retrieved photographs, not generated images** — a correct top-1
is pixel-identical to the stimulus, which is a retrieval hit rather than a
reconstruction. It is also a coarse discriminator: arms a few percent apart often give
identical rows, so read the statistics for the size of any difference.

**SDXL unCLIP decoder** (`--decoder sdxl_unclip`): the paper's decoder, and real
generation. 18 GB download, needs Stability's `sgm`, ~3-5 s/image. Frozen and identical
across arms, so it adds no between-arm variance and every statistic works without it.
Worth it on an A100; skip on a free T4.

In [ ]:
main(["recon", "--config", CONFIG])

# real generated images — uncomment on an A100 with disk to spare
# main(["recon", "--config", CONFIG, "--decoder", "sdxl_unclip",
#       "--arm", "frozen", "lora_r16", "full", "--n_images", "32"])

## 11. Evaluate

Per-image metrics. CLIP-space metrics always (cosine, two-way identification, retrieval
percentile); the eight MindEye image metrics as well if reconstructions exist.

In [ ]:
main(["evaluate", "--config", CONFIG])

## 12. Compare

Paired statistics against the full fine-tune: BCa bootstrap intervals, Wilcoxon with
Holm correction, Cohen's d_z, TOST equivalence, and the retention ratio.

Expect a difference that is *statistically detectable but practically negligible* for a
well-chosen rank. Both facts get reported, because with ~1,000 paired test images you
can detect gaps far below anything that matters.

In [ ]:
main(["compare", "--config", CONFIG])

## 13. Report

In [ ]:
main(["report", "--config", CONFIG])

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

report = Path(os.environ["MINDEYE_LORA_ROOT"]) / "results/reports/REPORT.md"
display(Markdown(report.read_text()))

In [ ]:
from IPython.display import Image
figdir = Path(os.environ["MINDEYE_LORA_ROOT"]) / "results/figures"
for p in sorted(figdir.glob("*.png")):
    print(p.name)
    display(Image(str(p)))

---

## Reading the report

Check the **frozen→full headroom** column first. If it is tiny, the shared-subject model
was already nearly sufficient, neither method had room to differ, and nothing else in
the report means much.

Then the **retention ratio** — the fraction of the achievable gain each arm recovered —
and the **equivalence** column, which is the positive claim. A non-significant p-value
alone never establishes equivalence.

## Poking at things

```python
main(["train", "--config", CONFIG, "--arm", "lora_r16", "--seed", "0"])  # one run
main(["verify", "--config", CONFIG, "--tree"])                           # every Linear layer
main(["compare", "--config", CONFIG, "--equivalence_fraction", "0.1"])   # stricter margin
main(["train", "--config", CONFIG, "--ignore-memory-check"])             # override preflight
```